# Exploratory Pattern Discovery

This notebook uses **unsupervised statistical techniques** to discover patterns in the MallWorld data without pre-specifying hypotheses. The goal is to let the data reveal its own structure.

## Techniques Used
1. **Correspondence Analysis** - Find associations between categorical variables
2. **Cluster Analysis** - Group dreams/locations by similarity
3. **Association Rule Mining** - Find "if X then Y" patterns
4. **Principal Component Analysis** - Reduce dimensionality, find latent factors
5. **Network Analysis** - Examine connection structure

## Research Integrity Note
We report what the data shows. Patterns discovered here may or may not align with any theoretical framework. We will clearly distinguish between:
- **Statistical patterns** (what the algorithms find)
- **Interpretation** (what these patterns might mean)

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.manifold import TSNE
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load data
data_dir = Path('../reports/data')
df_connections = pd.read_csv(data_dir / 'connections.csv')
df_locations = pd.read_csv(data_dir / 'locations.csv')
df_dreams = pd.read_csv(data_dir / 'dreams_meta.csv')
df_interactions = pd.read_csv(data_dir / 'interactions.csv')
df_entities = pd.read_csv(data_dir / 'entities.csv')

print(f"Dreams: {len(df_dreams):,}")
print(f"Locations: {len(df_locations):,}")
print(f"Connections: {len(df_connections):,}")
print(f"Interactions: {len(df_interactions):,}")
print(f"Entities: {len(df_entities):,}")

Dreams: 1,926
Locations: 8,707
Connections: 4,600
Interactions: 6,075
Entities: 3,704


---
## 1. Multiple Correspondence Analysis (MCA)

MCA finds associations between categorical variables simultaneously. It reveals which categories "go together" without us specifying which associations to look for.

In [3]:
# Check available columns first
print("Location columns:", df_locations.columns.tolist())
print("\\nConnection columns:", df_connections.columns.tolist())

# Prepare location data for MCA
# Select key categorical variables that exist
loc_vars = ['location_type', 'atmosphere', 'reality_stability']
df_loc_mca = df_locations[loc_vars].copy()

# Remove rows with missing data
df_loc_mca = df_loc_mca.dropna()
print(f"\\nLocations for MCA: {len(df_loc_mca):,}")

# Show value counts for each variable
for var in loc_vars:
    print(f"\\n{var}:")
    print(df_loc_mca[var].value_counts().head(8))

Location columns: ['dream_id', 'location_id', 'location_type', 'location_name', 'vertical_position', 'horizontal_position', 'light', 'light_temperature', 'time_of_day', 'state', 'atmosphere', 'reality_stability', 'crowding', 'crowd_behavior', 'intellectual_focus', 'cleanliness', 'privacy_status', 'water_presence', 'affective_response', 'somatic_response', 'is_familiar', 'is_accessible', 'dreamer_role', 'visit_order', 'raw_description', 'location_category', 'luxury_category']
\nConnection columns: ['dream_id', 'from_location_id', 'to_location_id', 'connection_type', 'transit_mode', 'direction', 'difficulty', 'mechanism_function', 'outcome', 'qualities']
\nLocations for MCA: 8,707
\nlocation_type:
location_type
other          2297
mall            879
mall_store      590
city_street     383
house           361
school          264
hotel           255
restaurant      192
Name: count, dtype: int64
\natmosphere:
atmosphere
not_mentioned    4401
neutral           865
threatening       713
eeri

In [4]:
# Simple Correspondence Analysis: Location Type × Atmosphere
# This finds which location types associate with which atmospheres

print("="*70)
print("CORRESPONDENCE ANALYSIS: Location Type × Atmosphere")
print("="*70)

# Create contingency table
contingency = pd.crosstab(df_loc_mca['location_type'], df_loc_mca['atmosphere'])

# Filter to location types with at least 30 occurrences
loc_counts = df_loc_mca['location_type'].value_counts()
common_locs = loc_counts[loc_counts >= 30].index
contingency = contingency.loc[contingency.index.isin(common_locs)]

print(f"\nContingency table shape: {contingency.shape}")
print(f"Location types: {len(contingency)}")
print(f"Atmospheres: {len(contingency.columns)}")

# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi-square test: χ² = {chi2:.2f}, df = {dof}, p = {p:.2e}")

# Compute residuals (observed - expected) / sqrt(expected)
# Positive residuals = more than expected, negative = less than expected
residuals = (contingency.values - expected) / np.sqrt(expected)
df_residuals = pd.DataFrame(residuals, index=contingency.index, columns=contingency.columns)

CORRESPONDENCE ANALYSIS: Location Type × Atmosphere

Contingency table shape: (51, 11)
Location types: 51
Atmospheres: 11

Chi-square test: χ² = 1677.05, df = 500, p = 6.75e-127


In [5]:
# Find the strongest associations (highest absolute residuals)
print("\n" + "="*70)
print("STRONGEST LOCATION-ATMOSPHERE ASSOCIATIONS")
print("(Standardized residuals > 2.0 indicate significant association)")
print("="*70)

associations = []
for loc in df_residuals.index:
    for atmos in df_residuals.columns:
        resid = df_residuals.loc[loc, atmos]
        if abs(resid) > 2.0:  # Significant association
            observed = contingency.loc[loc, atmos]
            associations.append((loc, atmos, resid, observed))

# Sort by absolute residual
associations.sort(key=lambda x: -abs(x[2]))

print("\nPOSITIVE associations (more than expected):")
print("-" * 60)
for loc, atmos, resid, obs in associations[:20]:
    if resid > 0:
        print(f"  {loc:25s} + {atmos:15s}  (z={resid:+.2f}, n={obs})")

print("\nNEGATIVE associations (less than expected):")
print("-" * 60)
for loc, atmos, resid, obs in associations:
    if resid < -2.0:
        print(f"  {loc:25s} - {atmos:15s}  (z={resid:+.2f}, n={obs})")


STRONGEST LOCATION-ATMOSPHERE ASSOCIATIONS
(Standardized residuals > 2.0 indicate significant association)

POSITIVE associations (more than expected):
------------------------------------------------------------
  bathroom_public           + uncomfortable    (z=+8.42, n=28)
  casino                    + chaotic          (z=+8.30, n=18)
  mountain                  + peaceful         (z=+7.95, n=10)
  basement                  + oppressive       (z=+7.89, n=11)
  beach                     + peaceful         (z=+7.36, n=15)
  restaurant                + welcoming        (z=+7.29, n=30)
  bathroom_locker_room      + uncomfortable    (z=+5.99, n=12)
  hotel_lobby               + welcoming        (z=+5.91, n=12)
  neighborhood              + nostalgic        (z=+5.71, n=11)
  carnival                  + chaotic          (z=+5.68, n=10)
  other                     + threatening      (z=+5.17, n=261)
  downtown                  + chaotic          (z=+4.88, n=18)
  city_street               +

In [6]:
# Correspondence Analysis: Location Type × Reality Stability
print("\n" + "="*70)
print("CORRESPONDENCE ANALYSIS: Location Type × Reality Stability")
print("="*70)

contingency_rs = pd.crosstab(df_loc_mca['location_type'], df_loc_mca['reality_stability'])
contingency_rs = contingency_rs.loc[contingency_rs.index.isin(common_locs)]

chi2, p, dof, expected = stats.chi2_contingency(contingency_rs)
print(f"\nChi-square test: χ² = {chi2:.2f}, df = {dof}, p = {p:.2e}")

residuals_rs = (contingency_rs.values - expected) / np.sqrt(expected)
df_residuals_rs = pd.DataFrame(residuals_rs, index=contingency_rs.index, columns=contingency_rs.columns)

print("\nSTRONGEST associations:")
for loc in df_residuals_rs.index:
    for stab in df_residuals_rs.columns:
        resid = df_residuals_rs.loc[loc, stab]
        if abs(resid) > 2.5:
            direction = "+" if resid > 0 else "-"
            print(f"  {loc:25s} {direction} {stab:15s}  (z={resid:+.2f})")


CORRESPONDENCE ANALYSIS: Location Type × Reality Stability

Chi-square test: χ² = 426.09, df = 250, p = 2.51e-11

STRONGEST associations:
  airport                   + not_mentioned    (z=+2.92)
  casino                    + plastic          (z=+3.30)
  hotel_lobby               + solid            (z=+2.69)
  house                     + shifting         (z=+5.32)
  mall_store                - shifting         (z=-3.43)
  mountain                  + hyper_real       (z=+2.83)
  mountain                  + solid            (z=+2.67)
  neighborhood              + plastic          (z=+4.15)
  restaurant_fast_food      + plastic          (z=+5.33)
  school                    - solid            (z=-2.76)
  underground               + decaying         (z=+3.55)
  unspecified               + not_mentioned    (z=+3.04)
  unspecified               - solid            (z=-3.48)


---
## 2. Cluster Analysis of Dreams

Can we identify distinct "types" of dreams based on their characteristics?

In [7]:
# Create dream-level features
print("="*70)
print("BUILDING DREAM FEATURE MATRIX")
print("="*70)

# For each dream, compute:
# - Number of locations
# - Number of connections
# - Dominant atmosphere
# - Dominant location type
# - Transit mode distribution
# - Direction distribution

dream_features = []

for dream_id in df_dreams['dream_id'].unique():
    locs = df_locations[df_locations['dream_id'] == dream_id]
    conns = df_connections[df_connections['dream_id'] == dream_id]
    inters = df_interactions[df_interactions['dream_id'] == dream_id]
    ents = df_entities[df_entities['dream_id'] == dream_id]
    
    features = {
        'dream_id': dream_id,
        'n_locations': len(locs),
        'n_connections': len(conns),
        'n_interactions': len(inters),
        'n_entities': len(ents),
    }
    
    # Atmosphere distribution
    if len(locs) > 0:
        atmos_counts = locs['atmosphere'].value_counts(normalize=True)
        for atmos in ['threatening', 'eerie', 'neutral', 'welcoming', 'peaceful', 'chaotic', 'oppressive']:
            features[f'atmos_{atmos}'] = atmos_counts.get(atmos, 0)
        
        # Reality stability
        stab_counts = locs['reality_stability'].value_counts(normalize=True)
        features['stability_solid'] = stab_counts.get('solid', 0)
        features['stability_shifting'] = stab_counts.get('shifting', 0)
    
    # Transit mode distribution
    if len(conns) > 0:
        mode_counts = conns['transit_mode'].value_counts(normalize=True)
        features['mode_directed'] = mode_counts.get('directed_active', 0)
        features['mode_passive'] = mode_counts.get('passive', 0)
        features['mode_fleeing'] = mode_counts.get('fleeing', 0)
        features['mode_wandering'] = mode_counts.get('wandering', 0)
        
        # Direction distribution
        dir_counts = conns['direction'].value_counts(normalize=True)
        features['dir_horizontal'] = dir_counts.get('horizontal', 0)
        features['dir_up'] = dir_counts.get('up', 0) + dir_counts.get('diagonal_up', 0)
        features['dir_down'] = dir_counts.get('down', 0) + dir_counts.get('diagonal_down', 0)
    
    dream_features.append(features)

df_dream_features = pd.DataFrame(dream_features)
df_dream_features = df_dream_features.fillna(0)

print(f"Dreams with features: {len(df_dream_features)}")
print(f"Features per dream: {len(df_dream_features.columns) - 1}")
print(f"\nFeatures: {list(df_dream_features.columns[1:])}")

BUILDING DREAM FEATURE MATRIX
Dreams with features: 1926
Features per dream: 20

Features: ['n_locations', 'n_connections', 'n_interactions', 'n_entities', 'atmos_threatening', 'atmos_eerie', 'atmos_neutral', 'atmos_welcoming', 'atmos_peaceful', 'atmos_chaotic', 'atmos_oppressive', 'stability_solid', 'stability_shifting', 'mode_directed', 'mode_passive', 'mode_fleeing', 'mode_wandering', 'dir_horizontal', 'dir_up', 'dir_down']


In [8]:
# Perform clustering
print("\n" + "="*70)
print("K-MEANS CLUSTERING OF DREAMS")
print("="*70)

# Select numeric features for clustering
feature_cols = [c for c in df_dream_features.columns if c != 'dream_id']
X = df_dream_features[feature_cols].values

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Find optimal k using elbow method
inertias = []
K_range = range(2, 10)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

print("Inertia by k:")
for k, inertia in zip(K_range, inertias):
    print(f"  k={k}: {inertia:.0f}")

# Use k=4 (based on elbow)
k_optimal = 4
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df_dream_features['cluster'] = kmeans.fit_predict(X_scaled)

print(f"\nCluster sizes (k={k_optimal}):")
print(df_dream_features['cluster'].value_counts().sort_index())


K-MEANS CLUSTERING OF DREAMS
Inertia by k:
  k=2: 34707
  k=3: 32557
  k=4: 31134
  k=5: 29178
  k=6: 27891
  k=7: 26691
  k=8: 25326
  k=9: 24172

Cluster sizes (k=4):
cluster
0     91
1    769
2    316
3    750
Name: count, dtype: int64


In [9]:
# Characterize each cluster
print("\n" + "="*70)
print("CLUSTER CHARACTERIZATION")
print("="*70)

for cluster in range(k_optimal):
    cluster_dreams = df_dream_features[df_dream_features['cluster'] == cluster]
    n = len(cluster_dreams)
    
    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster} (n={n}, {100*n/len(df_dream_features):.1f}%)")
    print("="*60)
    
    # Show mean values for key features
    print("\nStructure:")
    print(f"  Avg locations:    {cluster_dreams['n_locations'].mean():.1f}")
    print(f"  Avg connections:  {cluster_dreams['n_connections'].mean():.1f}")
    print(f"  Avg interactions: {cluster_dreams['n_interactions'].mean():.1f}")
    print(f"  Avg entities:     {cluster_dreams['n_entities'].mean():.1f}")
    
    print("\nAtmosphere (proportion):")
    for atmos in ['threatening', 'eerie', 'neutral', 'welcoming', 'peaceful', 'chaotic']:
        col = f'atmos_{atmos}'
        if col in cluster_dreams.columns:
            mean_val = cluster_dreams[col].mean()
            if mean_val > 0.05:  # Only show if > 5%
                print(f"  {atmos:15s}: {mean_val:.1%}")
    
    print("\nTransit mode (proportion):")
    for mode in ['directed', 'passive', 'fleeing', 'wandering']:
        col = f'mode_{mode}'
        if col in cluster_dreams.columns:
            print(f"  {mode:15s}: {cluster_dreams[col].mean():.1%}")
    
    print("\nDirection (proportion):")
    print(f"  horizontal:     {cluster_dreams['dir_horizontal'].mean():.1%}")
    print(f"  ascending:      {cluster_dreams['dir_up'].mean():.1%}")
    print(f"  descending:     {cluster_dreams['dir_down'].mean():.1%}")


CLUSTER CHARACTERIZATION

CLUSTER 0 (n=91, 4.7%)

Structure:
  Avg locations:    4.5
  Avg connections:  2.9
  Avg interactions: 3.1
  Avg entities:     1.5

Atmosphere (proportion):
  threatening    : 5.4%
  eerie          : 7.4%
  neutral        : 10.2%
  welcoming      : 5.2%
  chaotic        : 9.9%

Transit mode (proportion):
  directed       : 23.7%
  passive        : 2.9%
  fleeing        : 2.1%
  wandering      : 66.5%

Direction (proportion):
  horizontal:     76.5%
  ascending:      5.7%
  descending:     5.5%

CLUSTER 1 (n=769, 39.9%)

Structure:
  Avg locations:    2.4
  Avg connections:  0.7
  Avg interactions: 2.0
  Avg entities:     1.2

Atmosphere (proportion):
  threatening    : 8.7%
  eerie          : 6.3%
  neutral        : 6.0%
  welcoming      : 5.6%
  chaotic        : 6.6%

Transit mode (proportion):
  directed       : 2.4%
  passive        : 11.3%
  fleeing        : 1.1%
  wandering      : 0.0%

Direction (proportion):
  horizontal:     7.3%
  ascending:      6.2

---
## 3. Association Rule Mining

Find "if X then Y" patterns in the data. What features co-occur?

In [11]:
# Association rules for location features
print("="*70)
print("ASSOCIATION RULE MINING: Location Features")
print("="*70)

# Create binary features for each location
def compute_association_rules(df, feature_cols, min_support=0.01, min_confidence=0.5):
    """Simple association rule mining."""
    n_total = len(df)
    
    # Count single items
    item_counts = {}
    for col in feature_cols:
        if col not in df.columns:
            continue
        for val in df[col].unique():
            if pd.isna(val):
                continue
            item = f"{col}={val}"
            count = (df[col] == val).sum()
            support = count / n_total
            if support >= min_support:
                item_counts[item] = count
    
    # Find pairs with high confidence
    rules = []
    items = list(item_counts.keys())
    
    for i, item_a in enumerate(items):
        col_a, val_a = item_a.split('=', 1)
        mask_a = df[col_a] == val_a
        count_a = mask_a.sum()
        
        for item_b in items[i+1:]:
            col_b, val_b = item_b.split('=', 1)
            if col_a == col_b:  # Skip same column
                continue
                
            mask_b = df[col_b] == val_b
            count_both = (mask_a & mask_b).sum()
            
            if count_both < 10:  # Minimum frequency
                continue
            
            # A -> B
            conf_ab = count_both / count_a if count_a > 0 else 0
            if conf_ab >= min_confidence:
                support_ab = count_both / n_total
                # Lift = confidence / expected confidence
                expected_b = item_counts[item_b] / n_total
                lift = conf_ab / expected_b if expected_b > 0 else 0
                rules.append((item_a, item_b, support_ab, conf_ab, lift, count_both))
            
            # B -> A
            count_b = mask_b.sum()
            conf_ba = count_both / count_b if count_b > 0 else 0
            if conf_ba >= min_confidence:
                support_ba = count_both / n_total
                expected_a = item_counts[item_a] / n_total
                lift = conf_ba / expected_a if expected_a > 0 else 0
                rules.append((item_b, item_a, support_ba, conf_ba, lift, count_both))
    
    return rules

# Run association rule mining on locations (use only columns that exist)
loc_rules = compute_association_rules(
    df_locations, 
    ['location_type', 'atmosphere', 'reality_stability', 'light', 'crowding'],
    min_support=0.005,
    min_confidence=0.4
)

# Sort by lift (how much more likely than random)
loc_rules.sort(key=lambda x: -x[4])

print(f"\\nFound {len(loc_rules)} association rules")
print("\\nTop rules by LIFT (how much more than random):")
print("-" * 80)
print(f"{'Antecedent':<30} → {'Consequent':<25} Conf    Lift   n")
print("-" * 80)
for ant, cons, sup, conf, lift, n in loc_rules[:25]:
    print(f"{ant:<30} → {cons:<25} {conf:.1%}  {lift:.2f}x  {n}")

ASSOCIATION RULE MINING: Location Features
\nFound 232 association rules
\nTop rules by LIFT (how much more than random):
--------------------------------------------------------------------------------
Antecedent                     → Consequent                Conf    Lift   n
--------------------------------------------------------------------------------
crowding=packed                → atmosphere=chaotic        50.7%  8.91x  102
atmosphere=neutral             → reality_stability=solid   67.9%  2.00x  587
light=mixed                    → reality_stability=solid   63.2%  1.87x  117
light=bright_natural           → reality_stability=solid   62.9%  1.86x  212
crowding=sparse                → reality_stability=solid   62.5%  1.85x  292
light=bright_artificial        → reality_stability=solid   59.7%  1.76x  191
atmosphere=welcoming           → reality_stability=solid   59.2%  1.75x  231
light=dim                      → reality_stability=solid   57.9%  1.71x  176
atmosphere=peaceful     

In [12]:
# Association rules for connections
print("\n" + "="*70)
print("ASSOCIATION RULE MINING: Transit Features")
print("="*70)

conn_rules = compute_association_rules(
    df_connections,
    ['connection_type', 'transit_mode', 'direction', 'difficulty', 'outcome'],
    min_support=0.005,
    min_confidence=0.4
)

conn_rules.sort(key=lambda x: -x[4])

print(f"\nFound {len(conn_rules)} association rules")
print("\nTop rules by LIFT:")
print("-" * 80)
print(f"{'Antecedent':<30} → {'Consequent':<25} Conf    Lift   n")
print("-" * 80)
for ant, cons, sup, conf, lift, n in conn_rules[:25]:
    print(f"{ant:<30} → {cons:<25} {conf:.1%}  {lift:.2f}x  {n}")


ASSOCIATION RULE MINING: Transit Features

Found 136 association rules

Top rules by LIFT:
--------------------------------------------------------------------------------
Antecedent                     → Consequent                Conf    Lift   n
--------------------------------------------------------------------------------
difficulty=blocked             → outcome=failed            86.0%  80.78x  37
outcome=failed                 → difficulty=blocked        75.5%  80.78x  37
connection_type=teleport       → transit_mode=instant      95.2%  25.30x  59
connection_type=portal         → transit_mode=instant      60.0%  15.95x  33
transit_mode=struggle          → difficulty=difficult      79.1%  13.28x  53
connection_type=elevator       → transit_mode=passive      74.1%  6.58x  123
connection_type=flying         → direction=up              50.9%  4.95x  28
connection_type=stairs         → direction=up              49.5%  4.81x  189
connection_type=vehicle        → transit_mode=passive  

---
## 4. Principal Component Analysis

What are the latent dimensions underlying the data?

In [13]:
# PCA on dream features
print("="*70)
print("PRINCIPAL COMPONENT ANALYSIS: Dream Features")
print("="*70)

# Use the dream feature matrix from clustering
pca = PCA()
pca.fit(X_scaled)

print("\nVariance explained by each component:")
cumulative = 0
for i, var in enumerate(pca.explained_variance_ratio_[:10]):
    cumulative += var
    print(f"  PC{i+1}: {var:.1%} (cumulative: {cumulative:.1%})")

print(f"\nComponents needed for 80% variance: ", end="")
cumsum = np.cumsum(pca.explained_variance_ratio_)
n_components_80 = np.argmax(cumsum >= 0.80) + 1
print(n_components_80)

PRINCIPAL COMPONENT ANALYSIS: Dream Features

Variance explained by each component:
  PC1: 15.3% (cumulative: 15.3%)
  PC2: 8.6% (cumulative: 23.9%)
  PC3: 7.3% (cumulative: 31.1%)
  PC4: 6.7% (cumulative: 37.9%)
  PC5: 5.9% (cumulative: 43.8%)
  PC6: 5.8% (cumulative: 49.6%)
  PC7: 5.4% (cumulative: 55.0%)
  PC8: 5.1% (cumulative: 60.1%)
  PC9: 5.0% (cumulative: 65.1%)
  PC10: 4.9% (cumulative: 70.0%)

Components needed for 80% variance: 13


In [14]:
# Interpret principal components
print("\n" + "="*70)
print("PRINCIPAL COMPONENT INTERPRETATION")
print("="*70)

# Show loadings for first 4 components
for pc_idx in range(4):
    print(f"\nPC{pc_idx+1} ({pca.explained_variance_ratio_[pc_idx]:.1%} variance):")
    print("-" * 50)
    
    loadings = pca.components_[pc_idx]
    loading_pairs = list(zip(feature_cols, loadings))
    loading_pairs.sort(key=lambda x: -abs(x[1]))
    
    print("Strongest positive loadings:")
    for feat, load in loading_pairs[:5]:
        if load > 0.1:
            print(f"    {feat:<25}: {load:+.3f}")
    
    print("Strongest negative loadings:")
    for feat, load in loading_pairs[-5:]:
        if load < -0.1:
            print(f"    {feat:<25}: {load:+.3f}")


PRINCIPAL COMPONENT INTERPRETATION

PC1 (15.3% variance):
--------------------------------------------------
Strongest positive loadings:
    n_connections            : +0.470
    n_interactions           : +0.454
    n_locations              : +0.415
    n_entities               : +0.358
    dir_horizontal           : +0.325
Strongest negative loadings:

PC2 (8.6% variance):
--------------------------------------------------
Strongest positive loadings:
    mode_fleeing             : +0.494
    atmos_threatening        : +0.486
    n_entities               : +0.344
    n_interactions           : +0.216
Strongest negative loadings:

PC3 (7.3% variance):
--------------------------------------------------
Strongest positive loadings:
    mode_passive             : +0.483
    dir_down                 : +0.434
    dir_up                   : +0.394
Strongest negative loadings:

PC4 (6.7% variance):
--------------------------------------------------
Strongest positive loadings:
    stabilit

---
## 5. Entity Co-occurrence Analysis

Which types of entities appear together?

In [15]:
print("="*70)
print("ENTITY CO-OCCURRENCE ANALYSIS")
print("="*70)

# Get entity types per dream
dream_entities = df_entities.groupby('dream_id')['entity_type'].apply(set).to_dict()

# Count co-occurrences
entity_cooccur = Counter()
entity_counts = Counter()

for dream_id, entity_set in dream_entities.items():
    for ent in entity_set:
        entity_counts[ent] += 1
    for pair in combinations(sorted(entity_set), 2):
        entity_cooccur[pair] += 1

print(f"\nUnique entity types: {len(entity_counts)}")
print(f"Dreams with entities: {len(dream_entities)}")

print("\nMost common entity types:")
for ent, count in entity_counts.most_common(15):
    print(f"  {ent:<25}: {count}")

ENTITY CO-OCCURRENCE ANALYSIS

Unique entity types: 18
Dreams with entities: 1278

Most common entity types:
  crowd                    : 566
  stranger                 : 529
  authority                : 280
  threat                   : 190
  other                    : 167
  known_person             : 151
  family_member            : 138
  creature                 : 130
  friend                   : 72
  child                    : 52
  deceased                 : 33
  guide                    : 17
  watcher                  : 13
  shadow                   : 9
  coworker                 : 7


In [16]:
# Compute lift for entity pairs
print("\n" + "="*70)
print("ENTITY PAIR ASSOCIATIONS (by Lift)")
print("="*70)

n_dreams = len(dream_entities)
entity_lifts = []

for (ent_a, ent_b), cooccur in entity_cooccur.items():
    if cooccur < 10:  # Minimum frequency
        continue
    
    p_a = entity_counts[ent_a] / n_dreams
    p_b = entity_counts[ent_b] / n_dreams
    p_ab = cooccur / n_dreams
    
    expected = p_a * p_b
    lift = p_ab / expected if expected > 0 else 0
    
    entity_lifts.append((ent_a, ent_b, cooccur, lift))

entity_lifts.sort(key=lambda x: -x[3])

print("\nEntity pairs that co-occur MORE than expected:")
print("-" * 60)
for ent_a, ent_b, n, lift in entity_lifts[:20]:
    if lift > 1.5:
        print(f"  {ent_a:<20} + {ent_b:<20}: {lift:.2f}x (n={n})")

print("\nEntity pairs that co-occur LESS than expected:")
print("-" * 60)
for ent_a, ent_b, n, lift in reversed(entity_lifts[-10:]):
    if lift < 0.7:
        print(f"  {ent_a:<20} + {ent_b:<20}: {lift:.2f}x (n={n})")


ENTITY PAIR ASSOCIATIONS (by Lift)

Entity pairs that co-occur MORE than expected:
------------------------------------------------------------
  family_member        + friend              : 2.32x (n=18)
  child                + other               : 1.91x (n=13)
  child                + family_member       : 1.78x (n=10)
  authority            + child               : 1.58x (n=18)

Entity pairs that co-occur LESS than expected:
------------------------------------------------------------
  creature             + threat              : 0.67x (n=13)
  crowd                + deceased            : 0.68x (n=10)


---
## 6. Interaction Pattern Analysis

What types of interactions occur in what contexts?

In [17]:
print("="*70)
print("INTERACTION PATTERNS BY LOCATION TYPE")
print("="*70)

# Join interactions with location info
df_inter_loc = df_interactions.merge(
    df_locations[['location_id', 'location_type', 'atmosphere']],
    on='location_id',
    how='left'
)

# Cross-tab interaction type × location type
cross_inter_loc = pd.crosstab(df_inter_loc['interaction_type'], df_inter_loc['location_type'])

# Filter to common combinations
common_inter = df_inter_loc['interaction_type'].value_counts()
common_inter = common_inter[common_inter >= 50].index
common_loc = df_inter_loc['location_type'].value_counts()
common_loc = common_loc[common_loc >= 100].index

cross_filtered = cross_inter_loc.loc[
    cross_inter_loc.index.isin(common_inter),
    cross_inter_loc.columns.isin(common_loc)
]

chi2, p, dof, expected = stats.chi2_contingency(cross_filtered)
print(f"\nChi-square test: χ² = {chi2:.2f}, df = {dof}, p = {p:.2e}")

# Find strongest associations
residuals = (cross_filtered.values - expected) / np.sqrt(expected)
df_res = pd.DataFrame(residuals, index=cross_filtered.index, columns=cross_filtered.columns)

print("\nSTRONGEST Interaction-Location associations:")
print("-" * 60)
for inter in df_res.index:
    for loc in df_res.columns:
        z = df_res.loc[inter, loc]
        if abs(z) > 3.0:
            direction = "+" if z > 0 else "-"
            n = cross_filtered.loc[inter, loc]
            print(f"  {inter:<15} {direction} {loc:<20} (z={z:+.2f}, n={n})")

INTERACTION PATTERNS BY LOCATION TYPE

Chi-square test: χ² = 3495.86, df = 552, p = 0.00e+00

STRONGEST Interaction-Location associations:
------------------------------------------------------------
  conflict        - mall                 (z=-7.45, n=67485)
  conflict        + other                (z=+4.24, n=98300)
  escape          + basement             (z=+4.48, n=4183)
  escape          + bathroom             (z=+3.28, n=4799)
  escape          + hotel_ballroom       (z=+3.36, n=1619)
  escape          - mall                 (z=-14.94, n=136422)
  escape          + mall_theater         (z=+3.06, n=9932)
  escape          + other                (z=+8.78, n=202270)
  escape          + roof                 (z=+3.41, n=2361)
  escape          - school               (z=-3.16, n=28192)
  escape          + school_cafeteria     (z=+3.06, n=1960)
  escape          + school_library       (z=+3.35, n=1123)
  escape          + underground          (z=+3.92, n=5906)
  escape          - unspe

In [18]:
# Interaction outcomes by type
print("\n" + "="*70)
print("INTERACTION OUTCOMES BY TYPE")
print("="*70)

outcome_by_type = pd.crosstab(
    df_interactions['interaction_type'], 
    df_interactions['outcome'],
    normalize='index'
) * 100

# Filter to common interaction types
outcome_by_type = outcome_by_type.loc[outcome_by_type.index.isin(common_inter)]

print("\nSuccess rate by interaction type:")
if 'succeeded' in outcome_by_type.columns:
    success_rates = outcome_by_type['succeeded'].sort_values(ascending=False)
    for inter, rate in success_rates.items():
        n = df_interactions[df_interactions['interaction_type'] == inter].shape[0]
        print(f"  {inter:<20}: {rate:.1f}% success (n={n})")


INTERACTION OUTCOMES BY TYPE

Success rate by interaction type:
  transaction         : 45.9% success (n=220)
  task                : 37.8% success (n=815)
  navigation          : 37.0% success (n=1223)
  escape              : 34.6% success (n=752)
  other               : 32.9% success (n=149)
  conflict            : 32.8% success (n=338)
  social              : 30.9% success (n=795)
  observation         : 20.2% success (n=1315)
  search              : 16.0% success (n=468)


---
## 7. Summary: Discovered Patterns

In [19]:
print("="*70)
print("EXPLORATORY ANALYSIS: DISCOVERED PATTERNS SUMMARY")
print("="*70)

print("""
STATISTICAL PATTERNS DISCOVERED (facts from the data):
═══════════════════════════════════════════════════════════════════════

1. LOCATION-ATMOSPHERE ASSOCIATIONS (χ² = 1677, p < 10⁻¹²⁶)
─────────────────────────────────────────────────────────────────────
   Strong positive (more than expected):
   • bathroom_public → uncomfortable (z=+8.42)
   • casino → chaotic (z=+8.30)
   • mountain → peaceful (z=+7.95)
   • basement → oppressive (z=+7.89)
   • beach → peaceful (z=+7.36)
   • restaurant → welcoming (z=+7.29)
   
   Strong negative (less than expected):
   • mall_store ⊗ threatening (z=-4.55)
   • house ⊗ chaotic (z=-3.43)
   • school ⊗ threatening (z=-3.39)
   
2. LOCATION-STABILITY ASSOCIATIONS (χ² = 426, p < 10⁻¹⁰)
─────────────────────────────────────────────────────────────────────
   • house → shifting (z=+5.32)
   • fast_food restaurant → plastic (z=+5.33)  
   • neighborhood → plastic (z=+4.15)
   • underground → decaying (z=+3.55)
   • casino → plastic (z=+3.30)
   • mountain → hyper_real + solid (z=+2.83, +2.67)
   
3. DREAM CLUSTERS (k=4, by K-means)
─────────────────────────────────────────────────────────────────────
   Cluster 0 (5%):  "Wanderers" - 67% wandering mode, 77% horizontal
   Cluster 1 (40%): "Static" - few connections (0.7), 11% passive
   Cluster 2 (16%): "Threatened" - 23% threatening atmos, 33% fleeing
   Cluster 3 (39%): "Directed" - 80% directed_active, 14% ascending
   
4. TRANSIT ASSOCIATION RULES (Lift > 3.0)
─────────────────────────────────────────────────────────────────────
   • blocked → failed (86% confidence, 81x lift)
   • teleport → instant (95% confidence, 25x lift)
   • struggle → difficult (79% confidence, 13x lift)
   • elevator → passive (74% confidence, 7x lift)
   • flying/stairs → up (50% confidence, 5x lift)
   
5. PRINCIPAL COMPONENTS (latent dimensions)
─────────────────────────────────────────────────────────────────────
   PC1 (15%): "Complexity" - connections, interactions, locations
   PC2 (9%):  "Threat response" - fleeing mode + threatening atmosphere
   PC3 (7%):  "Vertical transit" - passive mode + up/down directions
   PC4 (7%):  "Stability" - solid + neutral/welcoming atmosphere
   
6. INTERACTION-LOCATION PATTERNS (χ² = 3496, p ≈ 0)
─────────────────────────────────────────────────────────────────────
   • escape ⊗ mall (z=-15), escape → basement/underground
   • social → mall (z=+23), social ⊗ parking/basement
   • transaction → mall_store (z=+6), transaction ⊗ mall (z=-18)
   • search → mall (z=+5)
   
7. INTERACTION SUCCESS RATES
─────────────────────────────────────────────────────────────────────
   Highest:  transaction (46%), task (38%), navigation (37%)
   Middle:   escape (35%), conflict (33%), social (31%)
   Lowest:   observation (20%), search (16%)

═══════════════════════════════════════════════════════════════════════
""")

print("KEY EMERGENT PATTERNS:")
print("-" * 70)
print("""
These patterns emerged from unsupervised techniques without hypothesis:

1. ATMOSPHERE-SPACE COHERENCE
   Natural spaces (mountain, beach) → peaceful
   Functional/commercial spaces → neutral
   Liminal/underground spaces → threatening/oppressive
   
2. REALITY STABILITY GRADIENT
   Physical/natural (mountain) → hyper_real/solid
   Domestic (house) → shifting
   Commercial simulacra (fast food, casino) → plastic
   Underground/decay → decaying
   
3. FOUR DREAM ARCHETYPES (from clustering)
   a) Wanderers: exploratory, horizontal, low agency
   b) Static: minimal movement, passive transport
   c) Threatened: high threat, escape behavior
   d) Directed: purposeful, ascending tendency
   
4. TRANSIT MODE-OUTCOME STRUCTURE
   Struggle → difficulty (79% confidence)
   Blocked → failure (86% confidence)
   Instant transit bypasses difficulty entirely
   
5. MALL AS SOCIAL SPACE (NOT transaction space)
   Social interactions: z=+22.82 in mall
   Transactions: z=-18.16 in mall (LESS than expected)
   Transactions happen in mall_stores, not the mall itself
""")

EXPLORATORY ANALYSIS: DISCOVERED PATTERNS SUMMARY

STATISTICAL PATTERNS DISCOVERED (facts from the data):
═══════════════════════════════════════════════════════════════════════

1. LOCATION-ATMOSPHERE ASSOCIATIONS (χ² = 1677, p < 10⁻¹²⁶)
─────────────────────────────────────────────────────────────────────
   Strong positive (more than expected):
   • bathroom_public → uncomfortable (z=+8.42)
   • casino → chaotic (z=+8.30)
   • mountain → peaceful (z=+7.95)
   • basement → oppressive (z=+7.89)
   • beach → peaceful (z=+7.36)
   • restaurant → welcoming (z=+7.29)

   Strong negative (less than expected):
   • mall_store ⊗ threatening (z=-4.55)
   • house ⊗ chaotic (z=-3.43)
   • school ⊗ threatening (z=-3.39)

2. LOCATION-STABILITY ASSOCIATIONS (χ² = 426, p < 10⁻¹⁰)
─────────────────────────────────────────────────────────────────────
   • house → shifting (z=+5.32)
   • fast_food restaurant → plastic (z=+5.33)  
   • neighborhood → plastic (z=+4.15)
   • underground → decaying (z=+3.

---
## 8. Location Type Transition Analysis

What location types tend to follow each other?

In [20]:
print("="*70)
print("LOCATION TRANSITION MATRIX")
print("="*70)

# Get location type sequences per dream
transitions = []
for dream_id in df_dreams['dream_id'].unique():
    locs = df_locations[df_locations['dream_id'] == dream_id].sort_values('visit_order')
    loc_types = locs['location_type'].tolist()
    for i in range(len(loc_types) - 1):
        transitions.append((loc_types[i], loc_types[i+1]))

# Count transitions
trans_counts = Counter(transitions)
print(f"Total transitions: {len(transitions)}")

# Get top location types
top_types = df_locations['location_type'].value_counts().head(12).index.tolist()

# Build transition matrix for top types
trans_matrix = pd.DataFrame(0, index=top_types, columns=top_types)
for (from_loc, to_loc), count in trans_counts.items():
    if from_loc in top_types and to_loc in top_types:
        trans_matrix.loc[from_loc, to_loc] = count

print("\nTransition matrix (top 12 location types):")
print(trans_matrix)

LOCATION TRANSITION MATRIX
Total transitions: 6781

Transition matrix (top 12 location types):
              other  mall  mall_store  city_street  house  school  hotel  \
other           723    79          53           76     49      31     32   
mall            143    59         143           18     14      19     15   
mall_store       94    50         138           10     13       9      8   
city_street      98    27           8           25     15       3      6   
house            96    15           3           12     32       6      4   
school           41    11           5            3      3      25      2   
hotel            35     8           1            9      4       0     15   
restaurant       23     9          13            6      4       4      7   
beach            37     5           3            5      8       2      4   
parking_lot      26    16           8           10      4       3      6   
neighborhood     17     8           1           10     24       1    

In [21]:
# Compute transition probabilities and compare to random
print("\n" + "="*70)
print("UNEXPECTED TRANSITIONS (compared to random)")
print("="*70)

# Row-normalize to get transition probabilities
row_sums = trans_matrix.sum(axis=1)
trans_prob = trans_matrix.div(row_sums, axis=0).fillna(0)

# Expected probability if random = column totals / total
col_totals = trans_matrix.sum(axis=0)
total_transitions = trans_matrix.sum().sum()
expected_prob = col_totals / total_transitions

# Find surprising transitions (observed >> expected or observed << expected)
surprises = []
for from_loc in top_types:
    for to_loc in top_types:
        obs = trans_prob.loc[from_loc, to_loc]
        exp = expected_prob[to_loc]
        n = trans_matrix.loc[from_loc, to_loc]
        if n >= 10:  # Minimum count
            ratio = obs / exp if exp > 0 else 0
            if ratio > 1.5 or ratio < 0.5:
                surprises.append((from_loc, to_loc, obs, exp, ratio, n))

surprises.sort(key=lambda x: -x[4])

print("\nTransitions MORE likely than random (ratio > 1.5):")
print("-" * 70)
for from_loc, to_loc, obs, exp, ratio, n in surprises[:15]:
    if ratio > 1.5:
        print(f"  {from_loc:15s} → {to_loc:15s}: {ratio:.2f}x expected (n={n})")

print("\nTransitions LESS likely than random (ratio < 0.5):")
print("-" * 70)
for from_loc, to_loc, obs, exp, ratio, n in reversed(surprises[-10:]):
    if ratio < 0.5:
        print(f"  {from_loc:15s} → {to_loc:15s}: {ratio:.2f}x expected (n={n})")


UNEXPECTED TRANSITIONS (compared to random)

Transitions MORE likely than random (ratio > 1.5):
----------------------------------------------------------------------
  school          → school         : 6.38x expected (n=25)
  neighborhood    → house          : 5.52x expected (n=24)
  restaurant      → restaurant     : 5.50x expected (n=21)
  hotel           → beach          : 4.86x expected (n=12)
  hotel           → hotel          : 4.64x expected (n=15)
  mall_store      → mall_store     : 3.04x expected (n=138)
  house           → house          : 3.00x expected (n=32)
  mall            → mall_store     : 2.48x expected (n=143)
  neighborhood    → city_street    : 2.06x expected (n=10)
  house           → neighborhood   : 2.03x expected (n=10)
  parking_lot     → mall           : 1.95x expected (n=16)
  parking_lot     → city_street    : 1.85x expected (n=10)
  city_street     → city_street    : 1.81x expected (n=25)
  mall_store      → parking_lot    : 1.51x expected (n=16)

Tra

---
## 9. Atmosphere Sequences Within Dreams

Do atmospheres tend to change in predictable ways during a dream?

In [22]:
print("="*70)
print("ATMOSPHERE TRAJECTORY ANALYSIS")
print("="*70)

# Define atmosphere valence (negative to positive)
atmos_valence = {
    'oppressive': -3,
    'threatening': -2,
    'eerie': -1,
    'uncomfortable': -1,
    'chaotic': -1,
    'wrong': -1,
    'neutral': 0,
    'not_mentioned': 0,
    'welcoming': 1,
    'nostalgic': 1,
    'peaceful': 2,
}

# Calculate atmosphere trajectory per dream
dream_trajectories = []
for dream_id in df_dreams['dream_id'].unique():
    locs = df_locations[df_locations['dream_id'] == dream_id].sort_values('visit_order')
    if len(locs) < 2:
        continue
    
    atmos_seq = [atmos_valence.get(a, 0) for a in locs['atmosphere']]
    
    # First half vs second half
    mid = len(atmos_seq) // 2
    first_half = np.mean(atmos_seq[:mid]) if mid > 0 else np.nan
    second_half = np.mean(atmos_seq[mid:]) if mid > 0 else np.nan
    
    # Change from start to end
    delta = atmos_seq[-1] - atmos_seq[0]
    
    dream_trajectories.append({
        'dream_id': dream_id,
        'first_half': first_half,
        'second_half': second_half,
        'delta': delta,
        'start': atmos_seq[0],
        'end': atmos_seq[-1]
    })

df_traj = pd.DataFrame(dream_trajectories)
df_traj = df_traj.dropna()

print(f"Dreams with trajectories: {len(df_traj)}")

# Test: Do atmospheres improve or worsen through dreams?
t_stat, p_value = stats.ttest_rel(df_traj['second_half'], df_traj['first_half'])
print(f"\nPaired t-test: First half vs Second half")
print(f"  First half mean:  {df_traj['first_half'].mean():.3f}")
print(f"  Second half mean: {df_traj['second_half'].mean():.3f}")
print(f"  t = {t_stat:.3f}, p = {p_value:.4f}")

if p_value < 0.05:
    direction = "WORSEN" if df_traj['second_half'].mean() < df_traj['first_half'].mean() else "IMPROVE"
    print(f"\n  → Atmospheres statistically {direction} during dreams")
else:
    print("\n  → No significant change in atmosphere trajectory")

# Distribution of start vs end atmospheres
print("\n" + "-"*50)
print("Start atmosphere distribution:")
start_dist = df_traj['start'].value_counts(normalize=True)
for val, pct in start_dist.items():
    label = [k for k, v in atmos_valence.items() if v == val][0] if val in atmos_valence.values() else str(val)
    print(f"  {label}: {pct:.1%}")

print("\nEnd atmosphere distribution:")
end_dist = df_traj['end'].value_counts(normalize=True)
for val, pct in end_dist.items():
    label = [k for k, v in atmos_valence.items() if v == val][0] if val in atmos_valence.values() else str(val)
    print(f"  {label}: {pct:.1%}")

ATMOSPHERE TRAJECTORY ANALYSIS
Dreams with trajectories: 1593

Paired t-test: First half vs Second half
  First half mean:  -0.337
  Second half mean: -0.404
  t = -3.348, p = 0.0008

  → Atmospheres statistically WORSEN during dreams

--------------------------------------------------
Start atmosphere distribution:
  neutral: 59.1%
  eerie: 19.8%
  welcoming: 8.7%
  threatening: 8.0%
  oppressive: 2.6%
  peaceful: 1.6%

End atmosphere distribution:
  neutral: 61.5%
  eerie: 18.4%
  threatening: 10.6%
  welcoming: 4.1%
  oppressive: 3.3%
  peaceful: 2.1%


---
## 10. Network Centrality Analysis

Which locations are "hubs" in the dream network?

In [23]:
import networkx as nx

print("="*70)
print("LOCATION TYPE NETWORK ANALYSIS")
print("="*70)

# Build network from transitions
G = nx.DiGraph()
for (from_loc, to_loc), count in trans_counts.items():
    if from_loc in top_types and to_loc in top_types:
        G.add_edge(from_loc, to_loc, weight=count)

# Compute centrality measures
in_degree = dict(G.in_degree(weight='weight'))
out_degree = dict(G.out_degree(weight='weight'))
pagerank = nx.pagerank(G, weight='weight')
betweenness = nx.betweenness_centrality(G, weight='weight')

print("\nIn-Degree (destinations - where dreams go TO):")
print("-" * 50)
for loc, deg in sorted(in_degree.items(), key=lambda x: -x[1])[:10]:
    print(f"  {loc:20s}: {deg:4d}")

print("\nOut-Degree (origins - where dreams go FROM):")
print("-" * 50)
for loc, deg in sorted(out_degree.items(), key=lambda x: -x[1])[:10]:
    print(f"  {loc:20s}: {deg:4d}")

print("\nPageRank (overall importance in network):")
print("-" * 50)
for loc, pr in sorted(pagerank.items(), key=lambda x: -x[1])[:10]:
    print(f"  {loc:20s}: {pr:.3f}")

print("\nBetweenness (bridge locations):")
print("-" * 50)
for loc, bc in sorted(betweenness.items(), key=lambda x: -x[1])[:10]:
    print(f"  {loc:20s}: {bc:.3f}")

LOCATION TYPE NETWORK ANALYSIS

In-Degree (destinations - where dreams go TO):
--------------------------------------------------
  other               : 1366
  mall_store          :  377
  mall                :  289
  city_street         :  191
  house               :  171
  restaurant          :  118
  school              :  107
  hotel               :  102
  parking_lot         :   88
  neighborhood        :   79

Out-Degree (origins - where dreams go FROM):
--------------------------------------------------
  other               : 1187
  mall                :  463
  mall_store          :  365
  city_street         :  219
  house               :  189
  school              :  111
  restaurant          :   98
  hotel               :   96
  parking_lot         :   86
  neighborhood        :   77

PageRank (overall importance in network):
--------------------------------------------------
  other               : 0.402
  mall_store          : 0.099
  mall                : 0.090
  city_st

---
## 11. Entity-Interaction Correlation Analysis

Which entity types are associated with which interactions?

In [24]:
print("="*70)
print("ENTITY-INTERACTION ANALYSIS")
print("="*70)

# Get entities per dream
dream_entity_types = df_entities.groupby('dream_id')['entity_type'].apply(set).to_dict()

# Get interactions per dream
dream_interaction_types = df_interactions.groupby('dream_id')['interaction_type'].apply(set).to_dict()

# Co-occurrence matrix
entity_types = df_entities['entity_type'].value_counts().head(10).index.tolist()
interaction_types = df_interactions['interaction_type'].value_counts().head(8).index.tolist()

cooccur = pd.DataFrame(0, index=entity_types, columns=interaction_types)

for dream_id in df_dreams['dream_id']:
    ents = dream_entity_types.get(dream_id, set())
    inters = dream_interaction_types.get(dream_id, set())
    for ent in ents:
        if ent in entity_types:
            for inter in inters:
                if inter in interaction_types:
                    cooccur.loc[ent, inter] += 1

print("Entity-Interaction co-occurrence counts:")
print(cooccur)

# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(cooccur)
print(f"\nChi-square test: χ² = {chi2:.2f}, df = {dof}, p = {p:.2e}")

# Calculate residuals
residuals = (cooccur.values - expected) / np.sqrt(expected)
df_res = pd.DataFrame(residuals, index=cooccur.index, columns=cooccur.columns)

print("\nSTRONGEST Entity-Interaction associations:")
print("-" * 60)
for ent in df_res.index:
    for inter in df_res.columns:
        z = df_res.loc[ent, inter]
        if abs(z) > 2.5:
            direction = "+" if z > 0 else "-"
            n = cooccur.loc[ent, inter]
            print(f"  {ent:15s} {direction} {inter:15s} (z={z:+.2f}, n={n})")

ENTITY-INTERACTION ANALYSIS
Entity-Interaction co-occurrence counts:
               observation  navigation  task  social  escape  search  \
stranger               300         251   243     340     187     143   
crowd                  360         290   238     277     221     137   
authority              161         141   142     130     148      75   
threat                  99          95    69      59     156      44   
known_person            87          63    64     103      44      46   
other                  116          70    75      83      74      49   
family_member           84          79    61      79      51      50   
creature                89          62    62      49      49      30   
friend                  39          36    36      50      29      27   
child                   37          28    30      34      27      18   

               conflict  transaction  
stranger            127           90  
crowd               138           61  
authority           1

---
## 12. Comprehensive Pattern Summary

In [25]:
print("="*80)
print("COMPREHENSIVE EXPLORATORY ANALYSIS SUMMARY")
print("="*80)

print("""
═══════════════════════════════════════════════════════════════════════════════
DISCOVERED PATTERNS - STATISTICAL FACTS
═══════════════════════════════════════════════════════════════════════════════

1. ATMOSPHERE-LOCATION COHERENCE (χ² = 1677, p < 10⁻¹²⁷)
   ─────────────────────────────────────────────────────────────────────────
   The data shows SYSTEMATIC associations between location types and atmospheres:
   
   NATURAL SPACES → POSITIVE ATMOSPHERES
   • Mountain → peaceful (z=+7.95)
   • Beach → peaceful (z=+7.36)
   
   SOCIAL/COMMERCIAL SPACES → NEUTRAL
   • Restaurant → welcoming (z=+7.29)
   • Mall_store → neutral (z=+3.91)
   
   LIMINAL/UNDERGROUND SPACES → NEGATIVE ATMOSPHERES
   • Basement → oppressive (z=+7.89)
   • Public bathroom → uncomfortable (z=+8.42)
   • Casino → chaotic (z=+8.30)

2. REALITY STABILITY GRADIENT (χ² = 426, p < 10⁻¹⁰)
   ─────────────────────────────────────────────────────────────────────────
   Location types have systematic relationships with reality stability:
   
   PHYSICAL/NATURAL → HYPER-REAL/SOLID
   • Mountain: hyper_real (z=+2.83), solid (z=+2.67)
   
   DOMESTIC/FAMILIAR → SHIFTING
   • House → shifting (z=+5.32)
   
   COMMERCIAL SIMULACRA → PLASTIC
   • Fast food restaurant → plastic (z=+5.33)
   • Casino → plastic (z=+3.30)
   • Neighborhood → plastic (z=+4.15)
   
   UNDERGROUND → DECAYING
   • Underground → decaying (z=+3.55)

3. DREAM ARCHETYPES (4 clusters from K-means)
   ─────────────────────────────────────────────────────────────────────────
   Cluster 0 (5%):  WANDERERS - 67% wandering mode, 77% horizontal
   Cluster 1 (40%): STATIC - minimal movement (0.7 connections), passive
   Cluster 2 (16%): THREATENED - 23% threatening, 33% fleeing
   Cluster 3 (39%): DIRECTED - 80% directed_active, 14% ascending

4. ATMOSPHERE TRAJECTORY (t = -3.35, p = 0.0008)
   ─────────────────────────────────────────────────────────────────────────
   Atmospheres WORSEN during dreams (first half: -0.34 → second half: -0.40)
   
   Start distribution: welcoming 8.7% → End distribution: welcoming 4.1%
   Start distribution: threatening 8.0% → End distribution: threatening 10.6%

5. LOCATION TYPE TRANSITIONS
   ─────────────────────────────────────────────────────────────────────────
   SELF-LOOPS (persistence within type):
   • School → School: 6.38x random
   • Restaurant → Restaurant: 5.50x random
   • Hotel → Hotel: 4.64x random
   
   COHERENT PATHWAYS:
   • Neighborhood → House: 5.52x random
   • Hotel → Beach: 4.86x random
   • Mall → Mall_store: 2.48x random
   • Parking_lot → Mall: 1.95x random

6. NETWORK CENTRALITY
   ─────────────────────────────────────────────────────────────────────────
   HUBS (PageRank):
   • Other (0.40), Mall_store (0.10), Mall (0.09), City_street (0.07)
   
   BRIDGES (Betweenness):
   • Apartment (0.36), Neighborhood (0.35), School (0.26)
   
   → Domestic spaces (apartment, neighborhood) serve as BRIDGES
   → Commercial spaces (mall, mall_store) serve as HUBS

7. ENTITY-INTERACTION ASSOCIATIONS (χ² = 211, p < 10⁻¹⁷)
   ─────────────────────────────────────────────────────────────────────────
   • Threat → Escape (z=+8.0), Conflict (z=+2.9), NOT social (z=-4.2)
   • Stranger → Social (z=+3.7), NOT escape (z=-2.7)
   • Known_person → Social (z=+3.0)
   
   → Entity type PREDICTS interaction type

8. TRANSIT GRAMMAR (from association rules)
   ─────────────────────────────────────────────────────────────────────────
   • Blocked → Failed (86% confidence, 81x lift)
   • Teleport → Instant mode (95% confidence, 25x lift)
   • Struggle → Difficult (79% confidence, 13x lift)
   • Flying/Stairs → Up direction (50% confidence, 5x lift)
   • Elevator/Vehicle → Passive mode (74%/52% confidence)

═══════════════════════════════════════════════════════════════════════════════
PRINCIPAL COMPONENTS (latent structure)
═══════════════════════════════════════════════════════════════════════════════

PC1 (15%): COMPLEXITY - n_connections, n_interactions, n_locations
PC2 (9%):  THREAT-RESPONSE - fleeing + threatening atmosphere
PC3 (7%):  VERTICAL TRANSIT - passive mode + up/down directions
PC4 (7%):  GROUNDEDNESS - solid stability + neutral/welcoming atmosphere

→ Data has 4 interpretable latent dimensions explaining 38% of variance
→ Need 13 components for 80% variance (high dimensionality)

═══════════════════════════════════════════════════════════════════════════════
""")

print("CRITICAL OBSERVATION:")
print("="*80)
print("""
These patterns emerged from UNSUPERVISED techniques without hypothesis specification.
The data reveals internal structure that was not imposed by the analysis method.

Key finding: The dream world has COHERENT INTERNAL LOGIC:
1. Locations have characteristic atmospheres
2. Reality stability follows a gradient (natural → domestic → commercial → underground)
3. Dreams follow predictable trajectories (atmosphere worsens)
4. Transitions between locations are non-random
5. Entity presence predicts interaction type
6. Transit mechanisms follow grammatical rules

This coherence exists INDEPENDENTLY of any theoretical framework.
What it MEANS is a separate interpretive question.
""")

COMPREHENSIVE EXPLORATORY ANALYSIS SUMMARY

═══════════════════════════════════════════════════════════════════════════════
DISCOVERED PATTERNS - STATISTICAL FACTS
═══════════════════════════════════════════════════════════════════════════════

1. ATMOSPHERE-LOCATION COHERENCE (χ² = 1677, p < 10⁻¹²⁷)
   ─────────────────────────────────────────────────────────────────────────
   The data shows SYSTEMATIC associations between location types and atmospheres:

   NATURAL SPACES → POSITIVE ATMOSPHERES
   • Mountain → peaceful (z=+7.95)
   • Beach → peaceful (z=+7.36)

   SOCIAL/COMMERCIAL SPACES → NEUTRAL
   • Restaurant → welcoming (z=+7.29)
   • Mall_store → neutral (z=+3.91)

   LIMINAL/UNDERGROUND SPACES → NEGATIVE ATMOSPHERES
   • Basement → oppressive (z=+7.89)
   • Public bathroom → uncomfortable (z=+8.42)
   • Casino → chaotic (z=+8.30)

2. REALITY STABILITY GRADIENT (χ² = 426, p < 10⁻¹⁰)
   ─────────────────────────────────────────────────────────────────────────
   Location types